# Direct Download to Google Drive
This notebook allows you to mount your Google Drive and directly download files using a URL.

## 1. Mount Google Drive
Run this cell to mount your Google Drive to the Colab environment so that downloaded files can be saved directly there.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Set Link and Download
Provide the direct download `url` and the `output_path` in Google Drive where you want to save it.

In [ ]:
import os
import requests
import ipywidgets as widgets
from IPython.display import display, clear_output
from tqdm.notebook import tqdm
import urllib.parse
import shutil

# 1. Create UI Widgets
url_input = widgets.Text(
    value='',
    placeholder='Enter direct download URL',
    description='URL:',
    layout=widgets.Layout(width='80%')
)

path_input = widgets.Text(
    value='/content/drive/MyDrive/Downloads',
    placeholder='Enter destination folder',
    description='Dest:',
    layout=widgets.Layout(width='80%')
)

download_button = widgets.Button(
    description='Start Download',
    button_style='success',
    icon='download'
)

output = widgets.Output()

# 2. Define Download Functionality
def download_file(b):
    with output:
        clear_output(wait=True)
        url = url_input.value.strip()
        final_dest_dir = path_input.value.strip()
        
        if not url:
            print("❌ Please enter a valid download URL.")
            return
            
        if not os.path.exists(final_dest_dir):
            try:
                os.makedirs(final_dest_dir)
            except Exception as e:
                 print(f"❌ Error creating destination directory. Is Google Drive mounted? Error: {e}")
                 return
            
        try:
            # Send initial request to get metadata
            response = requests.get(url, stream=True)
            response.raise_for_status()
            
            # Try to get filename from URL
            parsed_url = urllib.parse.urlparse(url)
            filename = os.path.basename(parsed_url.path)
            if not filename or filename == '':
                # If no filename in URL, try to get from headers
                if 'content-disposition' in response.headers:
                    import re
                    d = response.headers['content-disposition']
                    fname = re.findall("filename=(.+)", d)
                    if len(fname) > 0:
                        filename = fname[0].strip('"')
                
                if not filename:
                     filename = 'downloaded_file'
            
            # Download to Colab's local volatile storage first to prevent Drive timeouts
            temp_dir = '/content/temp_downloads'
            if not os.path.exists(temp_dir):
                os.makedirs(temp_dir)
                
            temp_file_path = os.path.join(temp_dir, filename)
            final_file_path = os.path.join(final_dest_dir, filename)
            
            # Remove temporary file if it somehow exists from a failed previous attempt
            if os.path.exists(temp_file_path):
                os.remove(temp_file_path)
            
            total_size = int(response.headers.get('content-length', 0))
            
            print(f"⬇️ Downloading to temporary storage first...")
            
            # Download with progress bar to local storage
            with open(temp_file_path, 'wb') as f, tqdm(
                desc=filename,
                total=total_size,
                unit='B',
                unit_scale=True,
                unit_divisor=1024,
            ) as bar:
                for chunk in response.iter_content(chunk_size=1024 * 1024): # 1MB chunks
                    if chunk:
                        size = f.write(chunk)
                        bar.update(size)
            
            print(f"🔄 Moving to final destination: {final_file_path}")
            
            # Remove final destination file if it exists to overwrite it without errors
            if os.path.exists(final_file_path):
                os.remove(final_file_path)
                
            shutil.move(temp_file_path, final_file_path)
            print("✅ Download and move completed successfully!")
            print("Ready for the next download.")
            
            # Clear the URL input to be ready for the next link natively
            url_input.value = ''
            
        except Exception as e:
            print(f"❌ Error downloading file: {e}")

download_button.on_click(download_file)

# 3. Display the Interface
display(widgets.VBox([url_input, path_input, download_button, output]))